# 🏛️ Observatorio Ciudadano de Contratación Pública (SECOP II)
### Inteligencia Artificial y Datos Abiertos para Explorar Contratos Estatales, Consultar la Ley y Anticipar Prórrogas

---

## 🌍 El Propósito del Proyecto
En Colombia, la contratación pública mueve una parte esencial del presupuesto del Estado. Aunque toda la información se publica a diario como datos abiertos en **SECOP II**, analizar millones de registros y cruzarlos con la legislación vigente sigue siendo una barrera enorme para ciudadanos, periodistas de investigación, veedurías y equipos de control interno.

Este proyecto nace para cerrar esa brecha a través de dos herramientas prácticas conectadas a una misma fuente de datos reales:

1. **Asistente de Consulta Ciudadana (RAG + SQL):** Permite hacer preguntas en lenguaje cotidiano y obtener respuestas que combinan cifras reales de contratos con los artículos exactos de la ley colombiana, citando siempre de dónde sale cada dato.
2. **Modelo Predictivo de Riesgo de Prórrogas:** Analiza las condiciones iniciales con las que se firma un contrato para estimar la probabilidad de que termine necesitando más tiempo del planeado (`dias_adicionados > 0`), ayudando a priorizar qué procesos merecen seguimiento preventivo.

> ⚠️ **Nota sobre el uso responsable:** Este observatorio identifica patrones estadísticos para orientar la revisión humana. Una prórroga o una alerta no equivale a una irregularidad ni reemplaza una auditoría oficial; cada caso debe contrastarse directamente en su expediente público mediante el enlace `urlproceso`.

> 📜 **Fuente y licencia de los datos:** Construido sobre datos abiertos de *«SECOP II – Contratos Electrónicos»* (Colombia Compra Eficiente, `datos.gov.co`) bajo licencia **CC BY-SA 4.0**. Proyecto independiente sin afiliación institucional.

--- 
## ⚙️ Punto de Partida: Preparación del Entorno
Antes de conectarnos a las fuentes oficiales, preparamos las rutas del proyecto, cargamos las credenciales de forma privada desde variables de entorno (`.env`) y definimos los parámetros base de nuestra consulta para que cualquier persona pueda reproducir el análisis.

In [ ]:
# Recarga automática de módulos en src/observatorio/ al editar archivos .py
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Asegurar que `src/` esté en sys.path incluso si se abre desde `notebooks/`
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from observatorio.config import (
    CLEAN_CSV_PATH,
    RANDOM_SEED,
    RAW_CSV_PATH,
    RAW_METADATA_PATH,
    SECOP_DEPARTAMENTO,
    SECOP_FECHA_FIN,
    SECOP_FECHA_INICIO,
    SQLITE_DB_PATH,
)
from observatorio.data.ingestion import COLUMNAS_SELECCIONADAS, descargar_subconjunto_secop
from observatorio.data.cleaning import (
    generar_resumen_validacion,
    limpiar_contratos_secop,
    obtener_diccionario_columnas,
)
from observatorio.data.database import ejecutar_consulta_segura, guardar_en_sqlite

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("✅ Entorno configurado correctamente.")
print(f"📍 Departamento objetivo : {SECOP_DEPARTAMENTO}")
print(f"📅 Rango de fechas       : {SECOP_FECHA_INICIO}  -->  {SECOP_FECHA_FIN}")
print(f"📄 Ruta CSV Original     : {RAW_CSV_PATH}")
print(f"🧹 Ruta CSV Limpio       : {CLEAN_CSV_PATH}")
print(f"🗄️ Ruta Base SQLite      : {SQLITE_DB_PATH}")

---
# 📦 PARTE 1: De los Datos Abiertos a una Base Confiable (SECOP II)

## 1.1 Conexión Directa con Datos Abiertos Colombia
En lugar de trabajar con archivos estáticos descargados a mano, consultamos directamente la API oficial de **Datos Abiertos (`datos.gov.co`)**. Acotamos la descarga a un departamento y una ventana de tiempo concreta, registrando la fecha exacta de corte (ya que SECOP II cambia todos los días) y guardando una copia intacta del archivo original (`secop_contratos_original.csv`) como respaldo para auditoría.

In [ ]:
# Cambia FORZAR_DESCARGA = True cuando quieras volver a consultar la API de datos.gov.co
FORZAR_DESCARGA = False
MAX_REGISTROS_DESCARGA = 50000  # Ajustable entre 20.000 y 100.000 según alcance (Sección 3)

if RAW_CSV_PATH.exists() and RAW_METADATA_PATH.exists() and not FORZAR_DESCARGA:
    print(f"📂 Cargando CSV original existente desde: {RAW_CSV_PATH}")
    df_raw = pd.read_csv(RAW_CSV_PATH, low_memory=False)
    metadatos_descarga = json.loads(RAW_METADATA_PATH.read_text(encoding="utf-8"))
else:
    print("🌐 Descargando subconjunto desde la API Socrata (datos.gov.co)...")
    df_raw, metadatos_descarga = descargar_subconjunto_secop(
        departamento=SECOP_DEPARTAMENTO,
        fecha_inicio=SECOP_FECHA_INICIO,
        fecha_fin=SECOP_FECHA_FIN,
        batch_size=10000,
        max_records=MAX_REGISTROS_DESCARGA,
        guardar_csv=True,
        ruta_csv=RAW_CSV_PATH,
        ruta_metadatos=RAW_METADATA_PATH,
    )
    print(f"✅ CSV original guardado en: {RAW_CSV_PATH}")

print("\n📋 Metadatos de trazabilidad de la descarga:")
for k, v in metadatos_descarga.items():
    if k != "columnas_descargadas":
        print(f"  - {k}: {v}")

print(f"\n🔎 Vista previa del CSV Original ({df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas):")
display(df_raw.head(5))

## 1.2 Curaduría de Datos y Protección de Privacidad
Los registros administrativos crudos suelen incluir ruido, formatos inconsistentes y datos sensibles. En este paso preparamos una versión limpia (`secop_contratos_limpio.csv`) aplicando tres criterios:
- **Privacidad por diseño:** Excluimos documentos de identidad, direcciones, cuentas bancarias y anonimizamos los nombres de contratistas cuando corresponden a personas naturales, manteniendo visibles únicamente las razones sociales de empresas (`NIT`).
- **Calidad y consistencia:** Eliminamos registros duplicados por `id_contrato`, estandarizamos fechas y valores monetarios, y extraemos el enlace directo al proceso original.
- **Contraste transparente:** Comparamos el archivo original frente al archivo limpio para verificar qué cambió exactamente durante el proceso.

In [ ]:
# Ejecutar limpieza y guardar el archivo CSV limpio en data/processed/secop_contratos_limpio.csv
df_clean = limpiar_contratos_secop(
    df_raw,
    guardar_csv=True,
    ruta_csv_limpio=CLEAN_CSV_PATH,
)

print(f"✅ CSV limpio guardado en: {CLEAN_CSV_PATH}")

# 1. Tabla comparativa de validación (CSV Original vs CSV Limpio)
print("\n📊 Tabla de Validación Comparativa (Original vs. Limpio):")
df_resumen_validacion = generar_resumen_validacion(df_raw, df_clean)
display(df_resumen_validacion)

# 2. Verificación visual de la transformación en columnas clave
cols_comparacion = [
    "id_contrato",
    "fecha_de_firma",
    "valor_del_contrato",
    "dias_adicionados",
    "tipodocproveedor",
    "proveedor_adjudicado",
    "urlproceso",
]
cols_presentes = [c for c in cols_comparacion if c in df_clean.columns]
print("\n🔎 Muestra del CSV Limpio en columnas auditadas:")
display(df_clean[cols_presentes].head(5))

# 3. Diccionario de las 25 columnas seleccionadas
print("\n📖 Diccionario de Columnas del Proyecto:")
display(obtener_diccionario_columnas())

## 1.3 Construcción de la Base de Datos de Consulta Rápida (`SQLite`)
Con el dataset curado y validado, lo estructuramos dentro de una base de datos relacional local (`secop_contratos.sqlite`). Esta base sirve como fuente única tanto para responder preguntas sobre cifras en el asistente conversacional como para alimentar el entrenamiento del modelo predictivo.

In [ ]:
# Volcar el DataFrame limpio en SQLite y crear índices
total_en_sqlite = guardar_en_sqlite(df_clean, db_path=SQLITE_DB_PATH)

assert total_en_sqlite == len(df_clean), (
    f"Discrepancia de conteo: SQLite ({total_en_sqlite}) != CSV Limpio ({len(df_clean)})"
)
print(f"✅ Base de datos SQLite creada e indexada en: {SQLITE_DB_PATH}")
print(f"✔️ Conteo verificado: {total_en_sqlite:,} contratos en SQLite == {len(df_clean):,} en CSV limpio.")

# Prueba rápida del ejecutor SQL en modo solo lectura (RF-8)
consulta_prueba = """
SELECT modalidad_de_contratacion,
       COUNT(*) AS total_contratos,
       ROUND(AVG(valor_del_contrato), 0) AS valor_promedio_cop
FROM contratos
GROUP BY modalidad_de_contratacion
ORDER BY total_contratos DESC
"""
df_sql_test, sql_ejecutado = ejecutar_consulta_segura(consulta_prueba, db_path=SQLITE_DB_PATH)
print(f"\n🔒 SQL ejecutado en modo solo lectura:\n{sql_ejecutado}")
display(df_sql_test)

---
# 🧠 PARTE 2: Asistente Híbrido de Contratación Pública (RAG Normativo + SQL)

## 2.1 Construyendo la Biblioteca Jurídica y Técnica
Un ciudadano no solo necesita saber *cuánto* se contrató, sino *qué dice la norma* al respecto. Aquí reunimos y estructuramos el marco regulatorio esencial de la contratación pública en Colombia (**Ley 80 de 1993**, **Ley 1150 de 2007**, **Decreto 1082 de 2015**, **Circular Externa Única** y los manuales/diccionarios de **Colombia Compra Eficiente**), organizando cada documento con su fuente, fecha y artículo para poder citarlo con precisión.

In [ ]:
# TODO: Lectura de archivos Markdown del corpus normativo y extracción/asignación de metadatos

## 2.2 Cómo Dividir la Ley sin Perder el Contexto (Chunking)
Si partimos un artículo de ley en pedazos demasiado pequeños, se pierde el sentido jurídico; si dejamos bloques demasiado largos, la búsqueda pierde precisión. En esta sección ponemos a prueba distintas estrategias de fragmentación respetando la estructura por artículos para elegir el equilibrio ideal.

In [ ]:
# TODO: Configurar al menos 3 tamaños en RecursiveCharacterTextSplitter
# TODO: Construir tabla comparativa de métricas de chunking y seleccionar el tamaño definitivo

## 2.3 Indexación Semántica en la Nube (`Qdrant Cloud` + `Gemini Embeddings`)
Transformamos cada fragmento normativo en representaciones vectoriales utilizando **Gemini Embedding 2** y los almacenamos en una base vectorial en la nube (**Qdrant Cloud**). De esta forma, el sistema puede encontrar el artículo pertinente aunque el usuario formule su pregunta con palabras cotidianas muy distintas al texto legal.

In [ ]:
# TODO: Inicializar cliente Qdrant Cloud y crear colección con VectorParams(size=768, distance=Distance.COSINE)
# TODO: Generar embeddings por lotes con Gemini Embedding 2, guardar respaldo local y hacer upsert en Qdrant

## 2.4 Conectando las Cifras (`SQL`) con las Normas (`RAG`)
Aquí unimos ambos mundos: diseñamos el buscador semántico sobre la normativa y el motor de consultas de solo lectura sobre la base de contratos. Así, el asistente puede decidir cuándo responder con estadísticas de la base de datos, cuándo citar un artículo de ley o cuándo combinar ambos en una sola respuesta.

In [ ]:
# TODO: Definir función de recuperación semántica Top-k sobre Qdrant con filtro opcional por documento
# TODO: Definir herramienta de consulta SQL de solo lectura (validación de SELECT único y límite de filas)

## 2.5 Poniendo a Prueba al Asistente: Precisión, Tiempos y Límites
Para asegurarnos de que el asistente sea confiable en escenarios reales, lo evaluamos con un banco de preguntas directas, preguntas en lenguaje coloquial, consultas que mezclan cifras con legislación y preguntas trampa sobre temas fuera del sistema (verificando que reconozca cuando no tiene la información en lugar de inventar respuestas).

In [ ]:
# TODO: Definir el banco de 10-15 preguntas de validación con su fuente esperada
# TODO: Ejecutar evaluación automatizada, calcular métricas (Top-1, Top-3, scores, latencia) y documentar análisis de fallos

---
# 📈 PARTE 3: Modelo Predictivo de Riesgo de Prórroga en Contratos

## 3.1 Entendiendo el Problema: ¿Qué Contratos Terminan Pidiendo Más Plazo?
Cuando un contrato público sufre prórrogas (`dias_adicionados > 0`), suele reflejar retos de planeación, cuellos de botella en la ejecución o complejidades propias del sector. Antes de entrenar cualquier algoritmo, exploramos los datos para entender qué tan frecuentes son las prórrogas y cómo varían según el tipo de contrato, la modalidad de selección y el monto involucrado.

In [ ]:
# TODO: Construcción y auditoría de la variable objetivo (dias_adicionados > 0)
# TODO: Gráficos exploratorios (distribución de clases, nulos, tasas de prórroga por modalidad/tipo y valor en escala log)

## 3.2 Regla de Oro: Usar Solo lo que se Sabe el Día de la Firma
Un error común al modelar datos públicos es incluir columnas que se actualizan meses después de firmado el contrato (como el estado de liquidación o los pagos acumulados), lo que produce modelos artificialmente perfectos pero inútiles en la vida real. Aquí seleccionamos exclusivamente las variables conocidas en el instante de la firma.

In [ ]:
# TODO: Definir listas explícitas de FEATURES_NUMERICAS y FEATURES_CATEGORICAS conocidas al momento de la firma
# TODO: Auditar ausencia de variables post-firma (data leakage)

## 3.3 Entrenar con el Pasado, Evaluar en el Futuro
Para simular cómo funcionaría esta herramienta en manos de una veeduría o una oficina de control interno, dividimos los contratos cronológicamente: entrenamos los modelos con el historial previo y los ponemos a prueba sobre los contratos más recientes.

In [ ]:
# TODO: Partición temporal (Train / Test split basado en fecha_de_firma)
# TODO: Definición del ColumnTransformer para variables numéricas y categóricas

## 3.4 Comparación de Modelos: De una Línea Base a Ensambles de Árboles
Comparamos un punto de referencia simple (`Baseline`) frente a distintos modelos clásicos de Machine Learning (**Regresión Logística**, **Random Forest** y **Gradient Boosting**), ajustando el desbalance natural de los datos para identificar cuál logra detectar mejor los contratos propensos a prórroga.

In [ ]:
# TODO: Configurar pipelines para Baseline, Regresión Logística, Random Forest y Gradient Boosting
# TODO: Entrenamiento, validación cruzada y búsqueda de hiperparámetros en el conjunto de entrenamiento

## 3.5 Resultados Reales: ¿Qué Aprendió el Modelo y Dónde se Equivoca?
Más allá de un puntaje global, analizamos qué factores pesan más en la estimación de riesgo (como la duración planeada inicial, la modalidad o el valor del contrato) y en qué tipos de procesos el modelo acierta o falla con mayor frecuencia.

In [ ]:
# TODO: Generar tabla comparativa de métricas en el conjunto de prueba temporal y graficar Matrices de Confusión / Curvas PR
# TODO: Calcular importancia de variables por permutación y analizar falsos positivos / falsos negativos por modalidad

---
# 🚀 PARTE 4: Del Análisis a la Aplicación Interactiva

## 4.1 Empaquetado del Modelo para Consulta en Tiempo Real
Exportamos el flujo completo del modelo ganador en un único archivo listo para producción (`pipeline_prorrogas.joblib`) y comprobamos que la aplicación web pueda cargarlo y generar estimaciones en una fracción de segundo.

In [ ]:
# TODO: Guardar artefacto del modelo con joblib.dump() y metadatos auxiliares en formato JSON
# TODO: Cargar el artefacto y verificar predicción unitaria (tiempo de respuesta y coincidencia exacta)

## 4.2 Transparencia, Alcance y Límites del Modelo
Para cerrar el flujo de desarrollo, dejamos documentadas las condiciones con las que fue entrenado el modelo, para qué sirve y cuáles son sus límites:
- **Para qué sirve:** Funciona como un filtro temprano para que periodistas, veedurías y equipos de auditoría decidan qué contratos revisar primero entre miles de procesos firmados.
- **Qué no debe hacerse con él:** No es un detector de corrupción ni un veredicto sobre la gestión de un contratista o entidad pública.
- **Limitaciones naturales:** Los datos tabulares de SECOP II resumen el contrato pero no contienen el texto detallado de los estudios previos ni las actas adjuntas, por lo que toda revisión debe culminar en el expediente electrónico original.